In [1]:
import matplotlib.pyplot as plt
import os
import plotly.graph_objects as go
import numpy as np
import sys
import os
sys.path.append(os.path.abspath('..'))
seed = 123
np.random.seed(seed)

def visualize_3d_interactively(emb1, emb2, labels=None, range=[-1,1]):
    fig = go.Figure()
    color_scale = 'Viridis'

    if labels is not None:
        unique_labels = np.unique(labels)
        for i, label in enumerate(unique_labels):
            indices = np.where(np.array(labels) == label)[0]
            fig.add_trace(go.Scatter3d(
                x=emb1[indices, 0], y=emb1[indices, 1], z=emb1[indices, 2],
                mode='markers',
                marker=dict(symbol='x', size=5, color=i, colorscale=color_scale),
                name=f'emb1 - {label}'
            ))
            fig.add_trace(go.Scatter3d(
                x=emb2[indices, 0], y=emb2[indices, 1], z=emb2[indices, 2],
                mode='markers',
                marker=dict(symbol='square', size=5, color=i, colorscale=color_scale),
                name=f'emb2 - {label}'
            ))
    else:
        fig.add_trace(go.Scatter3d(
            x=emb1[:, 0], y=emb1[:, 1], z=emb1[:, 2],
            mode='markers',
            marker=dict(symbol='x', size=5, color='blue'),
            name='emb1'
        ))
        fig.add_trace(go.Scatter3d(
            x=emb2[:, 0], y=emb2[:, 1], z=emb2[:, 2],
            mode='markers',
            marker=dict(symbol='square', size=5, color='red'),
            name='emb2'
        ))

    all_points = np.vstack([emb1, emb2])
    mins = all_points.min(axis=0)
    maxs = all_points.max(axis=0)
    padding = 0.05 * np.maximum(maxs - mins, 1e-6)

    fig.update_layout(
        title='3D Latent Space Visualization',
        scene=dict(
            xaxis_title='Dimension 1',
            yaxis_title='Dimension 2',
            zaxis_title='Dimension 3',
            xaxis=dict(range=[mins[0] - padding[0], maxs[0] + padding[0]] if range=='auto' else range),
            yaxis=dict(range=[mins[1] - padding[1], maxs[1] + padding[1]] if range=='auto' else range),
            zaxis=dict(range=[mins[2] - padding[2], maxs[2] + padding[2]] if range=='auto' else range),
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(x=0, y=1, traceorder='normal')
    )

    fig.show()


In [2]:
# Generate Nx3 array of random 3D points
def generate_random_points(num_points):
    return np.random.rand(num_points, 3)

def normalize_unit_sphere(points):
    return points / np.linalg.norm(points,ord=2, axis=1)[:, np.newaxis]

x = generate_random_points(100)
# Normalize into the unit sphere
x_unsph = normalize_unit_sphere(x)
# y = x + gap + np.random.normal(0, 0.1, x.shape)  # Add some noise to create a target variable
# move y from x of a given gap
translation = [2, -1, -0.5]  # Define a translation to create a linear relationship
y = x + translation   # Add some noise to create a target variable
y_unsph = normalize_unit_sphere(y)

In [3]:
print( np.linalg.norm(y))
print( np.linalg.norm(y_unsph[0]))

25.837297374607694
1.0


In [4]:
# Plot interactively  the points and the target variable
visualize_3d_interactively(x, y, labels=None, range='auto')

In [5]:
visualize_3d_interactively(x_unsph, y_unsph, labels=None)

In [6]:
# gap = mean of x_normalized - mean of y_normalized
gap = np.mean(x_unsph, axis=0) - np.mean(y_unsph, axis=0)
print("Gap between normalized x and y:", gap)

gamma = 1
x_tilde = x_unsph - gamma * (gap/2) # shifted but not on the unit sphere
y_tilde = y_unsph + gamma * (gap/2) # shifted but not on the unit sphere
visualize_3d_interactively(x_tilde, y_tilde, labels=None, range="auto")

Gap between normalized x and y: [-0.4599129   0.71344296  0.51878032]


In [7]:
x_tilde_normalized = normalize_unit_sphere(x_tilde)
y_tilde_normalized = normalize_unit_sphere(y_tilde)
visualize_3d_interactively(x_tilde_normalized, y_tilde_normalized)

In [8]:
from dataset.msrvtt.msrvtt_dataloaderv2 import (
    MSRVTTEmbeddingsDatasetV2,
    msrvtt_v2_collate_fn,
)
from torch.utils.data import DataLoader
import torch

g_v2 = torch.Generator().manual_seed(seed)



In [9]:
BATCH_SIZE_V2 = 256
NUM_WORKERS_V2 = 0

D_SUB_V2 = 256
N_FIT_V2 = 10_000
N_CLUSTERS_MSRVTT_V2 = 20
MAX_CLUSTER_SAMPLES_V2 = 5000
DIRECTION_V2 = 'text_to_vision'
precomputed_dir_v2 = '/mnt/media/emanuele/few_dimensions/dataset/msrvtt/ViT-B-32___laion2b_s34b_b79k_v2/precomputed_train'
precomputed_dir_test_v2 = '/mnt/media/emanuele/few_dimensions/dataset/msrvtt/ViT-B-32___laion2b_s34b_b79k_v2/precomputed_test'


ds_train_v2 = MSRVTTEmbeddingsDatasetV2(
    precomputed_dir_v2,
    split_name='train_shard',
    return_metadata=False,
)

ds_test_v2 = MSRVTTEmbeddingsDatasetV2(
    precomputed_dir_test_v2,
    split_name='test_shard',
    return_metadata=False,
)

train_loader_v2 = DataLoader(
    ds_train_v2,
    batch_size=BATCH_SIZE_V2,
    shuffle=True,
    num_workers=NUM_WORKERS_V2,
    collate_fn=msrvtt_v2_collate_fn,
    generator=g_v2,
)

test_loader_v2 = DataLoader(
    ds_test_v2,
    batch_size=BATCH_SIZE_V2,
    shuffle=False,
    num_workers=NUM_WORKERS_V2,
    collate_fn=msrvtt_v2_collate_fn,
)



[MSRVTTv2] 7010 unique videos from /mnt/media/emanuele/few_dimensions/dataset/msrvtt/ViT-B-32___laion2b_s34b_b79k_v2/precomputed_train | vision_emb=(7010, 512) | text_emb=(7010, 512) | num_classes=20
[MSRVTTv2] 1000 unique videos from /mnt/media/emanuele/few_dimensions/dataset/msrvtt/ViT-B-32___laion2b_s34b_b79k_v2/precomputed_test | vision_emb=(1000, 512) | text_emb=(1000, 512) | num_classes=20


In [10]:
iter_test = iter(test_loader_v2)
batch_test = next(iter_test)


In [11]:
batch_test[0].shape
# check that has magnitude 1
print("Magnitude of first batch sample:", np.linalg.norm(batch_test[0][0].numpy()))

Magnitude of first batch sample: 1.0
